In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Load Banknote Authentication Dataset
try:
    data = fetch_openml(data_id=1462, as_frame=True)
    X = data.data.select_dtypes(include=[np.number]).values.astype(float)
    y_raw = pd.Series(data.target).astype(str)
    y = pd.to_numeric(y_raw, errors="coerce").values.astype(int)
except Exception:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
    df = pd.read_csv(url, header=None)
    X = df.iloc[:, :4].values.astype(float)
    y = df.iloc[:, 4].values.astype(int)

feature_names = ["Variance", "Skewness", "Curtosis", "Entropy"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

class Perceptron:
    def __init__(self, learning_rate=0.01, max_epochs=1000):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.weights = None
        self.bias = 0.0
        self.epochs = 0
        self.converged = False
        self.error_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        self.weights = np.zeros(X.shape[1])
        self.bias = 0.0
        self.error_history = []
        self.converged = False

        for epoch in range(self.max_epochs):
            errors = 0

            for xi, target in zip(X, y):
                weighted_sum = np.dot(xi, self.weights) + self.bias
                prediction = 1 if weighted_sum >= 0 else 0
                error = target - prediction

                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1

            self.error_history.append(errors)
            self.epochs = epoch + 1

            if errors == 0:
                self.converged = True
                break

        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.weights + self.bias

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y) * 100

def class_name(value):
    return "Genuine" if int(value) == 0 else "Forged"

def train_model(indices=None, order="original", max_epochs=1000):
    if indices is None:
        indices = list(range(X_train.shape[1]))

    Xt = X_train[:, indices]
    yt = y_train.copy()

    if order == "reverse":
        Xt, yt = Xt[::-1], yt[::-1]
    elif order == "random":
        rng = np.random.default_rng(42)
        perm = rng.permutation(len(yt))
        Xt, yt = Xt[perm], yt[perm]
    elif order == "class-wise":
        perm = np.argsort(yt)
        Xt, yt = Xt[perm], yt[perm]

    net = Perceptron(learning_rate=0.01, max_epochs=max_epochs)
    net.fit(Xt, yt)
    return net

model = train_model()

# 1. Banknote Classification
def banknote_classification():
    outputs = model.decision_function(X_test)
    predictions = model.predict(X_test)

    print("\nBANKNOTE CLASSIFICATION")
    for i in range(len(y_test)):
        print(
            f"Sample {i+1:3d} | Actual: {class_name(y_test[i]):8s} | "
            f"Predicted: {class_name(predictions[i]):8s} | "
            f"Weighted Sum: {outputs[i]:.4f}"
        )

    print(f"\nAccuracy: {model.accuracy(X_test, y_test):.2f}%")
    print("Epochs:", model.epochs)
    print("Converged:", model.converged)
    print("Weights:", np.round(model.weights, 4))
    print("Bias:", round(model.bias, 4))

# 2. Decision Boundary Explorer
def decision_boundary_explorer():
    print("\nFEATURES")
    for i, name in enumerate(feature_names):
        print(f"{i+1}. {name}")

    a = int(input("Select first feature (1-4): ")) - 1
    b = int(input("Select second feature (1-4): ")) - 1

    if a not in range(4) or b not in range(4) or a == b:
        print("Select two different valid features.")
        return

    net = train_model([a, b])
    outputs = net.decision_function(X_test[:, [a, b]])
    predictions = net.predict(X_test[:, [a, b]])

    print("\nFinal Weights:", np.round(net.weights, 4))
    print("Bias:", round(net.bias, 4))
    print(
        f"Decision Boundary: {net.weights[0]:.4f}*{feature_names[a]} + "
        f"{net.weights[1]:.4f}*{feature_names[b]} + "
        f"{net.bias:.4f} = 0"
    )

    for i, output in enumerate(outputs):
        side = "Positive side" if output >= 0 else "Negative side"
        print(
            f"Sample {i+1:3d} | Output: {output:.4f} | "
            f"{side} | Predicted: {class_name(predictions[i])}"
        )

    print(f"Accuracy: {net.accuracy(X_test[:, [a, b]], y_test):.2f}%")

# 3. Feature Pair Separability Challenge
def feature_pair_challenge():
    from itertools import combinations

    results = []

    for pair in combinations(range(4), 2):
        net = train_model(list(pair))
        predictions = net.predict(X_test[:, list(pair)])

        results.append({
            "Feature Pair": f"{feature_names[pair[0]]} + {feature_names[pair[1]]}",
            "Epochs": net.epochs,
            "Converged": net.converged,
            "Accuracy (%)": net.accuracy(X_test[:, list(pair)], y_test),
            "Misclassified": int(np.sum(predictions != y_test))
        })

    df = pd.DataFrame(results).sort_values(
        ["Accuracy (%)", "Misclassified", "Epochs"],
        ascending=[False, True, True]
    )

    print("\nFEATURE PAIR RANKING")
    print(df.to_string(index=False))
    print("\nBest Feature Pair:")
    print(df.iloc[0].to_string())

# 4. Training Sample Order Challenge
def sample_order_challenge():
    results = []

    for order in ["original", "reverse", "random", "class-wise"]:
        net = train_model(order=order)
        results.append({
            "Sample Order": order,
            "Weights": np.round(net.weights, 4).tolist(),
            "Bias": round(net.bias, 4),
            "Epochs": net.epochs,
            "Accuracy (%)": net.accuracy(X_test, y_test),
            "Converged": net.converged
        })

    df = pd.DataFrame(results)
    print("\nTRAINING SAMPLE ORDER COMPARISON")
    print(df.to_string(index=False))

    boundaries = [tuple(row["Weights"]) + (row["Bias"],) for row in results]
    print("\nDifferent learned parameters:",
          "Yes" if len(set(boundaries)) > 1 else "No")

# 5. Borderline Sample Detector
def borderline_detector():
    outputs = model.decision_function(X_test)
    predictions = model.predict(X_test)
    indices = np.argsort(np.abs(outputs))[:5]

    records = []
    for i in indices:
        record = {
            "Sample": int(i + 1),
            "Actual": class_name(y_test[i]),
            "Predicted": class_name(predictions[i]),
            "Weighted Sum": outputs[i],
            "Boundary Distance": abs(outputs[i])
        }
        for j, name in enumerate(feature_names):
            record[name] = X_test_raw[i, j]
        records.append(record)

    print("\nTOP FIVE BORDERLINE SAMPLES")
    print(pd.DataFrame(records).round(4).to_string(index=False))

# 6. XOR Limitation Demonstrator
def xor_demonstrator():
    truth_tables = {
        "AND": (np.array([[0, 0], [0, 1], [1, 0], [1, 1]]),
                np.array([0, 0, 0, 1])),
        "OR": (np.array([[0, 0], [0, 1], [1, 0], [1, 1]]),
               np.array([0, 1, 1, 1])),
        "NAND": (np.array([[0, 0], [0, 1], [1, 0], [1, 1]]),
                 np.array([1, 1, 1, 0])),
        "XOR": (np.array([[0, 0], [0, 1], [1, 0], [1, 1]]),
                np.array([0, 1, 1, 0]))
    }

    for name, (Xt, yt) in truth_tables.items():
        net = Perceptron(learning_rate=0.1, max_epochs=100)
        net.fit(Xt, yt)
        predictions = net.predict(Xt)

        print(f"\n{name}")
        print("Weights:", np.round(net.weights, 4))
        print("Bias:", round(net.bias, 4))
        print("Epochs:", net.epochs)
        print("Converged:", net.converged)
        print("Predictions:", predictions)
        print("Actual:", yt)
        print("Misclassified:", int(np.sum(predictions != yt)))

        if name == "XOR":
            print("XOR is not linearly separable; one straight boundary cannot")
            print("separate its two classes. Misclassification pattern:")
            for epoch, errors in enumerate(net.error_history, 1):
                if errors > 0:
                    print(f"Epoch {epoch}: {errors} errors")
            if not net.converged:
                print("The perceptron did not converge within 100 epochs.")

# 7. Feature Removal Challenge
def feature_removal_challenge():
    baseline = model.accuracy(X_test, y_test)
    results = []

    for removed in range(4):
        indices = [i for i in range(4) if i != removed]
        net = train_model(indices)
        accuracy = net.accuracy(X_test[:, indices], y_test)

        results.append({
            "Removed Feature": feature_names[removed],
            "Accuracy (%)": accuracy,
            "Accuracy Drop (%)": baseline - accuracy,
            "Epochs": net.epochs,
            "Converged": net.converged
        })

    df = pd.DataFrame(results).sort_values(
        "Accuracy Drop (%)", ascending=False
    )

    print("\nFEATURE REMOVAL CHALLENGE")
    print(f"Baseline Accuracy: {baseline:.2f}%")
    print(df.round(4).to_string(index=False))
    print("Largest accuracy drop after removing:", df.iloc[0]["Removed Feature"])

# 8. Classification Report
def classification_report():
    results = []

    for order in ["original", "reverse", "random", "class-wise"]:
        net = train_model(order=order)
        outputs = net.decision_function(X_test)
        predictions = net.predict(X_test)

        results.append({
            "Selected Features": ", ".join(feature_names),
            "Sample Order": order,
            "Epochs": net.epochs,
            "Weights": np.round(net.weights, 4).tolist(),
            "Bias": round(net.bias, 4),
            "Accuracy (%)": net.accuracy(X_test, y_test),
            "Borderline Samples": int(np.sum(np.abs(outputs) < 0.1)),
            "Misclassified": int(np.sum(predictions != y_test)),
            "Converged": net.converged
        })

    print("\nPERCEPTRON CLASSIFICATION REPORT")
    print(pd.DataFrame(results).to_string(index=False))

    print("\nFEATURE PAIR SUMMARY")
    feature_pair_challenge()

    print("\nFEATURE REMOVAL SUMMARY")
    feature_removal_challenge()

# Menu
def main_menu():
    while True:
        print("\n" + "=" * 65)
        print(" PERCEPTRON DECISION BOUNDARY AND LINEAR SEPARABILITY SYSTEM")
        print("=" * 65)
        print("1. Banknote Classification using Perceptron")
        print("2. Decision Boundary Explorer")
        print("3. Feature Pair Separability Challenge")
        print("4. Training Sample Order Challenge")
        print("5. Borderline Sample Detector")
        print("6. XOR Limitation Demonstrator")
        print("7. Perceptron Feature Removal Challenge")
        print("8. Perceptron Classification Report")
        print("9. Exit")

        choice = input("\nEnter your choice (1-9): ").strip()

        if choice == "1":
            banknote_classification()
        elif choice == "2":
            decision_boundary_explorer()
        elif choice == "3":
            feature_pair_challenge()
        elif choice == "4":
            sample_order_challenge()
        elif choice == "5":
            borderline_detector()
        elif choice == "6":
            xor_demonstrator()
        elif choice == "7":
            feature_removal_challenge()
        elif choice == "8":
            classification_report()
        elif choice == "9":
            print("\nPROGRAM FINISHED")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 9.")

main_menu()


 PERCEPTRON DECISION BOUNDARY AND LINEAR SEPARABILITY SYSTEM
1. Banknote Classification using Perceptron
2. Decision Boundary Explorer
3. Feature Pair Separability Challenge
4. Training Sample Order Challenge
5. Borderline Sample Detector
6. XOR Limitation Demonstrator
7. Perceptron Feature Removal Challenge
8. Perceptron Classification Report
9. Exit
